# Hands-On Lab: Cotality Property Portfolio Intelligence with BigQuery AI & ML

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

> **📌 Demonstration Scope Notice:** Authored for live presenter workshop demonstration and sandbox validation (`Copyright 2026 Google LLC`, `SPDX-License-Identifier: Apache-2.0`). Complete formal open-source release review prior to distributing notebook source files externally.

## Overview

Property management, mortgage servicing, and real estate intelligence teams traditionally stitch together **five or more disconnected systems** to manage a residential portfolio: object storage and custom computer-vision pipelines for exterior/roof inspection photos, NLP microservices for resident maintenance triage, separate BI wikis for metric governance, standalone forecasting pipelines for regional rent outlooks, and custom tabular ML training jobs for automated rent valuation.

In this end-to-end hands-on lab, **100% of that property intelligence lifecycle executes in place inside BigQuery using GoogleSQL**—governed by a portable **Open Knowledge Format (OKF v0.2)** semantic bundle:

| Lab Step | Property Management Capability | BigQuery & Governance Primitives |
| :--- | :--- | :--- |
| **Step 1** | **Environment, Authentication & Storage Setup** | `google-cloud-bigquery`, `google-cloud-storage`, End-User / Connection Auth |
| **Step 2** | **Sample Property Tables & Unstructured Inspection Photos** | Partitioned & Clustered DDL, Cloud Storage PNGs, `ML.DESCRIBE_DATA` |
| **Step 3** | **Open Knowledge Format (OKF v0.2) Semantic Bundle** | `INFORMATION_SCHEMA` hydration, `Attested Computation`, Grounded `AI.GENERATE` |
| **Step 4** | **Multimodal Roof & Exterior Hazard Underwriting** | **`OBJ.MAKE_REF`**, `OBJ.FETCH_METADATA`, **`AI.GENERATE`** (`output_schema`) |
| **Step 5** | **Intelligent Work-Order Triage & Vendor Dispatch** | **`AI.IF`** (24-hr emergency filter) + **`AI.CLASSIFY`** (trade routing) |
| **Step 6** | **Zero-Shot Regional Rent Roll Forecasting** | **`AI.FORECAST`** + `AI.EVALUATE` (built-in **TimesFM** foundation model) |
| **Step 7** | **Zero-Shot Fair-Market Rent & Renewal Prediction** | **`AI.PREDICT`** *(Public Preview)* + `AI.EVALUATE` (**TabFM** tabular model) |
| **Step 8** | **Portfolio Executive Action Memo & Cleanup** | Cross-modal SQL synthesis (`AI.GENERATE`) + Key Findings & Cleanup |

> **🎙️ Live Demo Talk Track (Opening Hook — 60s):**
> *"Today we're walking through a real end-to-end Cotality property management workflow—without moving a single byte of data out to external ML clusters or custom ETL microservices. Starting from raw parcel tables, resident maintenance tickets, and exterior roof inspection photos in Cloud Storage, we'll package our business rules in Open Knowledge Format (OKF), inspect roof damage directly from SQL with `OBJ.MAKE_REF` and `AI.GENERATE`, triage emergency leaks with `AI.IF` and `AI.CLASSIFY`, forecast metro rent rolls zero-shot with `AI.FORECAST`, and price fair-market lease renewals zero-shot with `AI.PREDICT`."*

---
## Step 1: Install Dependencies, Set Project Name & Authenticate

First, we install/upgrade `google-cloud-bigquery`, `google-cloud-storage`, `pandas`, `Pillow` (for rendering crisp field-inspection images), `PyYAML` (for Open Knowledge Format validation), and plotting libraries.

> **🎙️ Live Demo Talk Track (Step 1):**
> *"Notice how lightweight our setup is. BigQuery's generative and foundation-model SQL functions (`AI.GENERATE`, `AI.IF`, `AI.CLASSIFY`, `AI.FORECAST`, and `AI.PREDICT`) run server-side inside BigQuery. Even better, BigQuery supports **end-user credentials** and **direct `OBJ.MAKE_REF(uri)` access**, so analysts can prototype multimodal queries immediately using their own IAM permissions—even inside VPC Service Controls perimeters—or optionally pass a delegated Cloud Resource Connection ID for scheduled production pipelines."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

# Install/upgrade BigQuery, Cloud Storage, imaging, YAML, and visualization libraries
!pip install --quiet --upgrade "google-cloud-bigquery>=3.25.0" "google-cloud-storage>=2.14.0" "pandas==2.2.3" matplotlib seaborn tabulate pillow pyyaml

import google.cloud.bigquery as bq
import google.cloud.storage as gcs
import pandas as pd
import PIL

print(f"✅ google-cloud-bigquery version : {bq.__version__}")
print(f"✅ google-cloud-storage version  : {gcs.__version__}")
print(f"✅ pandas version                : {pd.__version__}")
print(f"✅ Pillow version                : {PIL.__version__}")

### 1.1 Configure Project, Dataset & Cloud Storage Bucket

Enter your target Google Cloud **`PROJECT_ID`** and **`BUCKET_NAME`** in the form fields below and run the cell:
* In **Google Colab**, this cell prompts an interactive OAuth sign-in (`auth.authenticate_user()`) and binds your credentials to `PROJECT_ID`.
* In **Vertex AI Workbench / local Jupyter** with Application Default Credentials (ADC) configured, it automatically uses your active credentials.
* Leave `CONNECTION_ID` blank to execute multimodal and generative AI queries directly with **end-user credentials** (`roles/aiplatform.user` + `roles/storage.objectViewer`), or specify an optional BigQuery Cloud Resource Connection (e.g., `us.cotality-conn`) for delegated service-account execution.

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

import os
import subprocess
import google.auth
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage

# @markdown ### 🔧 Google Cloud Project & Storage Configuration
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
LOCATION = "US"  # @param ["US", "EU", "us-central1", "us-east1"]
DATASET_ID = "cotality_property_ai"  # @param {type:"string"}
BUCKET_NAME = ""  # @param {type:"string"}
CONNECTION_ID = ""  # @param {type:"string"}
BUCKET_PREFIX = "cotality_property_demo"

# Strip any accidental leading/trailing whitespace from copy-pasting
PROJECT_ID = PROJECT_ID.strip()
LOCATION = LOCATION.strip()
DATASET_ID = DATASET_ID.strip()
BUCKET_NAME = (BUCKET_NAME.strip() or f"{PROJECT_ID}-cotality-demo").lower()
CONNECTION_ID = CONNECTION_ID.strip()

if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "⚠️ Please set PROJECT_ID to your Google Cloud Project ID before running this cell."
    )

# 1. Authenticate in Google Colab (if running in Colab runtime)
try:
    from google.colab import auth
    print(f"🔐 Authenticating user in Google Colab for project '{PROJECT_ID}'...")
    auth.authenticate_user(project_id=PROJECT_ID)
    print("✅ Google Colab user authentication complete.")
except ImportError:
    print("ℹ️ Not running inside Google Colab; using Application Default Credentials (ADC).")

# 2. Set active project & ensure required APIs are enabled
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GCLOUD_PROJECT"] = PROJECT_ID
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=False)
subprocess.run(
    [
        "gcloud",
        "services",
        "enable",
        "bigquery.googleapis.com",
        "bigqueryconnection.googleapis.com",
        "aiplatform.googleapis.com",
        "storage.googleapis.com",
        f"--project={PROJECT_ID}",
    ],
    check=False,
)

# 3. Initialize BigQuery & Cloud Storage clients
credentials, _ = google.auth.default(quota_project_id=PROJECT_ID)
bq_client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials,
)
storage_client = storage.Client(
    project=PROJECT_ID,
    credentials=credentials,
)

# 4. Idempotently create BigQuery Dataset & Cloud Storage Bucket
dataset_fqn = f"{PROJECT_ID}.{DATASET_ID}"
dataset_ref = bigquery.Dataset(dataset_fqn)
dataset_ref.location = LOCATION
dataset_ref.description = (
    "Cotality Property Portfolio Intelligence lab dataset (structured + multimodal AI/ML)."
)
bq_client.create_dataset(dataset_ref, exists_ok=True)

bucket_location = LOCATION if LOCATION in ("US", "EU") else LOCATION.lower()
bucket_obj = storage_client.bucket(BUCKET_NAME)
if not bucket_obj.exists():
    try:
        bucket_obj = storage_client.create_bucket(bucket_obj, location=bucket_location)
        print(f"🪣 Created Cloud Storage bucket : gs://{BUCKET_NAME} ({bucket_location})")
    except Conflict:
        bucket_obj = storage_client.get_bucket(BUCKET_NAME)
else:
    bucket_obj = storage_client.get_bucket(BUCKET_NAME)

# Helper SQL snippets for optional Cloud Resource Connection
CONN_ARG_SQL = f", connection_id => '{CONNECTION_ID}'" if CONNECTION_ID else ""
MAKE_REF_CONN_SQL = f", '{CONNECTION_ID}'" if CONNECTION_ID else ""

print("-" * 74)
print(f"🚀 Authenticated Target Project : {PROJECT_ID}")
print(f"📍 BigQuery Execution Location  : {LOCATION}")
print(f"🗄️ Target BigQuery Dataset      : {dataset_fqn}")
print(f"🪣 Target Cloud Storage Bucket  : gs://{BUCKET_NAME}/{BUCKET_PREFIX}/")
print(
    f"🔑 Multimodal Execution Mode    : "
    f"{'Delegated Connection (' + CONNECTION_ID + ')' if CONNECTION_ID else 'Direct End-User Credentials'}"
)
print("-" * 74)

---
## Step 2: Generate Sample Property Datasets & Unstructured Inspection Images

To model an authentic Cotality residential & small-multifamily portfolio across **6 high-growth US metros** (`Austin, TX`, `Miami, FL`, `Denver, CO`, `Phoenix, AZ`, `Seattle, WA`, `Tampa, FL`), we generate both **unstructured visual inspection assets** in Cloud Storage and **four partitioned/clustered BigQuery tables**:

1. **Unstructured Cloud Storage Images (`gs://{BUCKET_NAME}/cotality_property_demo/images/*.png`)**:
   * Six field-inspection captures covering intact architectural shingle roofs, severe hail/wind shingle loss with exposed underlayment, interior ceiling drywall water stains, rooftop solar upgrades, cracked exterior stucco with overflowing gutters, and coastal elevated storm-shutter construction.
2. **Structured BigQuery Tables (`{PROJECT_ID}.cotality_property_ai`)**:
   * **`properties`** (`CLUSTER BY metro_area, property_type, risk_zone` — 180 managed properties with APN parcel IDs, structural attributes, peril risk scores, current monthly rent, operating expenses, and `gcs_image_uri`).
   * **`maintenance_requests`** (`PARTITION BY request_date CLUSTER BY property_id, status` — natural-language resident & field-inspector tickets).
   * **`regional_rent_history`** (`PARTITION BY month_date CLUSTER BY metro_area, property_type` — 84 months $\times$ 12 series = 1,008 monthly rent index observations).
   * **`property_valuation_comps`** (`CLUSTER BY metro_area, property_type` — 360 comparable property leases with 12 tabular features, `market_monthly_rent_usd`, and `lease_renewal_outcome`).

> **🎙️ Live Demo Talk Track (Step 2):**
> *"First, let's populate our sandbox with realistic property data—both structured and unstructured. Notice we apply BigQuery storage best practices right out of the gate: date partitioning on `maintenance_requests` and `regional_rent_history`, and multi-column clustering on `metro_area`, `property_type`, and `risk_zone`. At the same time, we upload six exterior and interior property inspection images to Cloud Storage so BigQuery can reason over pixels and parcel tables in a single query."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

import pathlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

LOCAL_IMG_DIR = pathlib.Path("cotality_demo_assets/images")
LOCAL_IMG_DIR.mkdir(parents=True, exist_ok=True)


def render_property_inspection_png(
    out_path: pathlib.Path,
    prop_id: str,
    banner_title: str,
    inspector_caption: str,
    scene_kind: str,
) -> None:
    """Renders a clear, high-contrast architectural property inspection PNG."""
    w, h = 800, 520
    img = Image.new("RGB", (w, h), "#E8F1F8")
    d = ImageDraw.Draw(img)

    if scene_kind == "interior_leak":
        # Interior room ceiling & wall with brown concentric water intrusion stain
        d.rectangle([0, 48, w, 240], fill="#F3F4F6")  # ceiling plane
        d.rectangle([0, 240, w, h - 64], fill="#E5E7EB")  # wall plane
        d.line([(0, 240), (w, 240)], fill="#9CA3AF", width=3)
        # Water damage rings + peeling drywall crack + active droplets
        d.ellipse([230, 85, 570, 215], fill="#D6BFA3", outline="#A67C52", width=3)
        d.ellipse([285, 105, 515, 195], fill="#B0895A", outline="#7C5329", width=2)
        d.ellipse([345, 125, 455, 175], fill="#7C5329")
        d.line([(310, 148), (395, 156), (475, 142)], fill="#2B1D0E", width=4)
        for dx in (375, 405, 432):
            d.ellipse([dx - 5, 220, dx + 5, 238], fill="#2563EB")
            d.ellipse([dx - 4, 265, dx + 4, 282], fill="#2563EB")
        d.rectangle([330, 390, 470, 445], fill="#60A5FA", outline="#1D4ED8", width=3)  # catch bucket
    else:
        # Exterior sky + lawn + house facade + pitched roof
        sky = "#CBD5E1" if scene_kind in ("hail_damage", "coastal_flood") else "#DBEAFE"
        d.rectangle([0, 48, w, 360], fill=sky)
        d.rectangle([0, 360, w, h - 64], fill="#86EFAC" if scene_kind != "coastal_flood" else "#93C5FD")

        base_top = 240 if scene_kind != "coastal_flood" else 205
        base_bot = 385 if scene_kind != "coastal_flood" else 325

        if scene_kind == "coastal_flood":
            # Elevated concrete pilings above coastal surge zone
            for px in (210, 320, 480, 590):
                d.rectangle([px - 12, 325, px + 12, 415], fill="#64748B", outline="#334155", width=2)

        # House exterior walls & windows
        wall_color = "#FEF3C7" if scene_kind == "stucco_crack" else "#F8FAFC"
        d.rectangle([170, base_top, 630, base_bot], fill=wall_color, outline="#475569", width=3)
        for wx in (225, 495):
            shutter_fill = "#94A3B8" if scene_kind == "coastal_flood" else "#BFDBFE"
            d.rectangle([wx, base_top + 32, wx + 80, base_top + 92], fill=shutter_fill, outline="#334155", width=3)
            if scene_kind == "coastal_flood":
                for sy in range(base_top + 40, base_top + 90, 10):
                    d.line([(wx + 4, sy), (wx + 76, sy)], fill="#475569", width=2)
        d.rectangle([372, base_bot - 85, 428, base_bot], fill="#78350F", outline="#451A03", width=2)

        # Pitched roof polygon
        roof_color = "#334155"
        if scene_kind == "coastal_flood":
            roof_color = "#0F766E"  # standing-seam coastal metal roof
        roof_poly = [(135, base_top), (400, base_top - 125), (665, base_top)]
        d.polygon(roof_poly, fill=roof_color, outline="#0F172A")

        # Shingle course lines
        for ry in range(base_top - 100, base_top, 18):
            half_span = int((ry - (base_top - 125)) * (265 / 125))
            d.line([(400 - half_span, ry), (400 + half_span, ry)], fill="#1E293B", width=2)

        if scene_kind == "hail_damage":
            # Missing torn shingle patches exposing dark felt underlayment & orange hail markers
            for (mx, my) in [(325, 165), (385, 185), (445, 160), (495, 198), (285, 202)]:
                d.rectangle([mx, my, mx + 44, my + 24], fill="#7F1D1D", outline="#FCA5A5", width=2)
            for (hx, hy) in [(305, 155), (360, 175), (420, 205), (470, 182), (530, 212), (350, 218)]:
                d.ellipse([hx - 6, hy - 6, hx + 6, hy + 6], outline="#F97316", width=3)

        elif scene_kind == "solar_modern":
            # Sleek blue-black photovoltaic solar array on intact roof
            d.polygon([(275, 175), (515, 175), (555, 225), (235, 225)], fill="#1E3A8A", outline="#93C5FD")
            for sx in range(275, 525, 45):
                d.line([(sx, 175), (sx + 10, 225)], fill="#93C5FD", width=2)
            d.line([(255, 200), (535, 200)], fill="#93C5FD", width=2)

        elif scene_kind == "stucco_crack":
            # Sagging clogged gutter overflow + diagonal foundation/stucco settlement crack
            d.line([(160, base_top + 3), (640, base_top + 12)], fill="#B45309", width=6)
            d.polygon([(580, base_top + 12), (610, base_top + 12), (625, base_bot)], fill="#93C5FD")
            d.line(
                [(330, base_bot), (345, base_bot - 35), (338, base_bot - 65), (358, base_bot - 98)],
                fill="#991B1B",
                width=4,
            )

    # Top & bottom field-inspection camera HUD overlays
    d.rectangle([0, 0, w, 48], fill="#0F172A")
    d.rectangle([0, h - 64, w, h], fill="#1E293B")
    font = ImageFont.load_default()
    d.text((16, 16), f"COTALITY INSPECTION CAM | {prop_id} | {banner_title}", fill="#F8FAFC", font=font)
    d.text((16, h - 48), f"FIELD OBSERVATION: {inspector_caption}", fill="#FDE68A", font=font)
    d.text((16, h - 24), "CAPTURE MODE: HIGH-RES RGB | VERIFIED PARCEL ASSET", fill="#94A3B8", font=font)
    img.save(out_path, format="PNG")


INSPECTION_SPECS = [
    {
        "property_id": "PROP-1001",
        "filename": "prop_1001_intact_shingle_roof.png",
        "banner_title": "AUSTIN, TX - ARCHITECTURAL SHINGLE ROOF (GRADE A)",
        "inspector_caption": "Clean 3-year architectural shingles, intact flashing, clear gutters, zero visible defects.",
        "scene_kind": "intact_roof",
    },
    {
        "property_id": "PROP-1002",
        "filename": "prop_1002_hail_damaged_roof.png",
        "banner_title": "DENVER, CO - SEVERE HAIL & WIND SHINGLE LOSS",
        "inspector_caption": "URGENT HAZARD: Multiple missing shingle tabs, exposed dark underlayment, hail impact bruising.",
        "scene_kind": "hail_damage",
    },
    {
        "property_id": "PROP-1003",
        "filename": "prop_1003_ceiling_water_intrusion.png",
        "banner_title": "TAMPA, FL - ACTIVE INTERIOR CEILING WATER INTRUSION",
        "inspector_caption": "URGENT HAZARD: Brown ceiling drywall saturation rings, cracking seam, active water dripping.",
        "scene_kind": "interior_leak",
    },
    {
        "property_id": "PROP-1004",
        "filename": "prop_1004_solar_modern_exterior.png",
        "banner_title": "PHOENIX, AZ - SOLAR-EQUIPPED ROOF & EXCELLENT EXTERIOR",
        "inspector_caption": "Permitted 8kW rooftop solar array on intact roof; sealed stucco envelope; zero hazards.",
        "scene_kind": "solar_modern",
    },
    {
        "property_id": "PROP-1005",
        "filename": "prop_1005_stucco_crack_gutter.png",
        "banner_title": "SEATTLE, WA - SAGGING GUTTER OVERFLOW & STUCCO CRACK",
        "inspector_caption": "MODERATE DEFECT: Sagging downspout water runoff staining facade + diagonal stucco settlement crack.",
        "scene_kind": "stucco_crack",
    },
    {
        "property_id": "PROP-1006",
        "filename": "prop_1006_coastal_elevated_home.png",
        "banner_title": "MIAMI, FL - ELEVATED PILINGS & HURRICANE SHUTTERS",
        "inspector_caption": "Fortified standing-seam metal roof, concrete flood pilings, intact storm shutters; good condition.",
        "scene_kind": "coastal_flood",
    },
]

uploaded_image_uris = {}
for spec in INSPECTION_SPECS:
    local_file = LOCAL_IMG_DIR / spec["filename"]
    render_property_inspection_png(
        out_path=local_file,
        prop_id=spec["property_id"],
        banner_title=spec["banner_title"],
        inspector_caption=spec["inspector_caption"],
        scene_kind=spec["scene_kind"],
    )
    gcs_rel_path = f"{BUCKET_PREFIX}/images/{spec['filename']}"
    blob = bucket_obj.blob(gcs_rel_path)
    blob.upload_from_filename(str(local_file), content_type="image/png")
    gcs_uri = f"gs://{BUCKET_NAME}/{gcs_rel_path}"
    uploaded_image_uris[spec["property_id"]] = gcs_uri
    print(f"✅ Uploaded {spec['property_id']} -> {gcs_uri}")

# Visual preview grid of all 6 uploaded property inspection images
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, spec in zip(axes.flat, INSPECTION_SPECS):
    img_arr = Image.open(LOCAL_IMG_DIR / spec["filename"])
    ax.imshow(img_arr)
    ax.set_title(f"{spec['property_id']}: {spec['banner_title'][:38]}", fontsize=9.5, fontweight="bold")
    ax.axis("off")
plt.suptitle("Unstructured Property Inspection Images Uploaded to Cloud Storage", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

import time

print("⏳ Materializing partitioned & clustered Cotality property tables in BigQuery...")
t0 = time.perf_counter()

SEED_SQL = f"""
-- 1. Properties Master Table (180 managed parcels, clustered for regional + peril filtering)
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.properties`
CLUSTER BY metro_area, property_type, risk_zone
AS
WITH raw_parcels AS (
  SELECT
    idx,
    FORMAT('PROP-%04d', 1000 + idx) AS property_id,
    FORMAT('APN-%02d-%04d-%02d', 10 + MOD(idx, 6) * 7, 1000 + idx, MOD(idx * 3, 97)) AS apn_parcel_id,
    CASE idx
      WHEN 1 THEN 'Austin, TX'
      WHEN 2 THEN 'Denver, CO'
      WHEN 3 THEN 'Tampa, FL'
      WHEN 4 THEN 'Phoenix, AZ'
      WHEN 5 THEN 'Seattle, WA'
      WHEN 6 THEN 'Miami, FL'
      ELSE ['Austin, TX', 'Miami, FL', 'Denver, CO', 'Phoenix, AZ', 'Seattle, WA', 'Tampa, FL'][OFFSET(MOD(idx, 6))]
    END AS metro_area,
    CASE
      WHEN idx IN (1, 2, 4, 6) THEN 'Single-Family'
      WHEN idx = 3 THEN 'Townhome'
      WHEN idx = 5 THEN 'Small Multifamily'
      ELSE ['Single-Family', 'Single-Family', 'Townhome', 'Small Multifamily'][OFFSET(MOD(idx, 4))]
    END AS property_type,
    1986 + MOD(ABS(FARM_FINGERPRINT(CAST(idx AS STRING))), 37) AS year_built,
    1250 + MOD(ABS(FARM_FINGERPRINT(CONCAT('sqft_', CAST(idx AS STRING)))), 1850) AS sqft
  FROM UNNEST(GENERATE_ARRAY(1, 180)) AS idx
)
SELECT
  property_id,
  apn_parcel_id,
  CONCAT(CAST(100 + idx * 14 AS STRING), ' Oakridge Parcel Way, ', metro_area) AS address,
  metro_area,
  property_type,
  year_built,
  2026 - year_built AS property_age_years,
  sqft,
  GREATEST(2, LEAST(5, CAST(ROUND(sqft / 650.0) AS INT64))) AS bedrooms,
  ROUND(1.5 + MOD(idx, 4) * 0.5, 1) AS bathrooms,
  CASE
    WHEN idx = 6 OR metro_area = 'Miami, FL' THEN 'Standing Seam Metal'
    WHEN metro_area = 'Phoenix, AZ' AND idx != 4 THEN 'Clay Tile'
    WHEN MOD(idx, 5) = 0 THEN 'Three-Tab Asphalt'
    ELSE 'Architectural Shingle'
  END AS roof_material,
  CASE idx
    WHEN 1 THEN 3
    WHEN 2 THEN 17
    WHEN 3 THEN 15
    WHEN 4 THEN 4
    WHEN 5 THEN 12
    WHEN 6 THEN 5
    ELSE 2 + MOD(ABS(FARM_FINGERPRINT(CONCAT('roof_', CAST(idx AS STRING)))), 19)
  END AS roof_age_years,
  CASE
    WHEN idx = 2 OR metro_area = 'Denver, CO' THEN 'Moderate-Hail'
    WHEN idx IN (3, 6) OR metro_area IN ('Miami, FL', 'Tampa, FL') THEN 'High-Wind-Flood'
    WHEN metro_area = 'Phoenix, AZ' AND MOD(idx, 3) = 0 THEN 'Moderate-Wildfire'
    ELSE 'Low'
  END AS risk_zone,
  ROUND(
    CASE idx
      WHEN 1 THEN 0.18
      WHEN 2 THEN 0.84
      WHEN 3 THEN 0.88
      WHEN 4 THEN 0.21
      WHEN 5 THEN 0.58
      WHEN 6 THEN 0.44
      ELSE 0.15 + (MOD(ABS(FARM_FINGERPRINT(CONCAT('haz_', CAST(idx AS STRING)))), 75) / 100.0)
    END,
    2
  ) AS hazard_risk_score,
  52 + MOD(ABS(FARM_FINGERPRINT(CONCAT('walk_', CAST(idx AS STRING)))), 44) AS walkability_score,
  (idx = 4 OR MOD(idx, 6) = 0) AS has_solar_upgrade,
  CASE
    WHEN MOD(idx, 19) = 0 THEN 'Vacant-Turn'
    WHEN MOD(idx, 9) = 0 THEN 'Notice-Given'
    ELSE 'Occupied'
  END AS occupancy_status,
  ROUND(
    (
      CASE metro_area
        WHEN 'Seattle, WA' THEN 2450
        WHEN 'Miami, FL' THEN 2550
        WHEN 'Denver, CO' THEN 2150
        WHEN 'Austin, TX' THEN 2050
        WHEN 'Tampa, FL' THEN 1980
        ELSE 1900
      END
      + sqft * 0.58
      -- Intentionally make ~25% of occupied units under-rented legacy leases so AI.PREDICT surfaces upside!
      - IF(MOD(idx, 4) = 1, 290.0, 0.0)
    ),
    0
  ) AS current_monthly_rent_usd,
  ROUND(5400 + sqft * 2.15 + IF(metro_area IN ('Miami, FL', 'Tampa, FL'), 2100, 650), 0) AS annual_operating_expense_usd,
  ROUND(240000 + sqft * 155 + IF(metro_area IN ('Seattle, WA', 'Miami, FL'), 95000, 25000), 0) AS estimated_asset_value_usd,
  CASE
    WHEN idx IN (2, 3) THEN 4
    WHEN idx = 5 THEN 2
    ELSE MOD(ABS(FARM_FINGERPRINT(CONCAT('clm_', CAST(idx AS STRING)))), 5)
  END AS prior_maintenance_claims_12m,
  CASE idx
    WHEN 1 THEN 'gs://{BUCKET_NAME}/{BUCKET_PREFIX}/images/prop_1001_intact_shingle_roof.png'
    WHEN 2 THEN 'gs://{BUCKET_NAME}/{BUCKET_PREFIX}/images/prop_1002_hail_damaged_roof.png'
    WHEN 3 THEN 'gs://{BUCKET_NAME}/{BUCKET_PREFIX}/images/prop_1003_ceiling_water_intrusion.png'
    WHEN 4 THEN 'gs://{BUCKET_NAME}/{BUCKET_PREFIX}/images/prop_1004_solar_modern_exterior.png'
    WHEN 5 THEN 'gs://{BUCKET_NAME}/{BUCKET_PREFIX}/images/prop_1005_stucco_crack_gutter.png'
    WHEN 6 THEN 'gs://{BUCKET_NAME}/{BUCKET_PREFIX}/images/prop_1006_coastal_elevated_home.png'
    ELSE NULL
  END AS gcs_image_uri
FROM raw_parcels;

-- 2. Maintenance Requests Table (Partitioned by request_date, Clustered by property_id & status)
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.maintenance_requests`
PARTITION BY request_date
CLUSTER BY property_id, status
AS
WITH curated_open_tickets AS (
  SELECT * FROM UNNEST([
    STRUCT('REQ-5001' AS request_id, 'PROP-1003' AS property_id, DATE '2026-10-28' AS request_date, 'After-Hours Hotline' AS reported_channel, 'OPEN' AS status,
      'Water is actively dripping from a brown stain in the living room ceiling drywall after heavy rain last night; bucket placed underneath.' AS tenant_description),
    STRUCT('REQ-5002', 'PROP-1002', DATE '2026-10-28', 'Inspector Mobile App', 'OPEN',
      'Post-hailstorm drone check shows 14+ missing architectural shingle tabs on south roof slope with exposed dark felt underlayment.'),
    STRUCT('REQ-5003', 'PROP-1004', DATE '2026-10-27', 'Resident Portal', 'OPEN',
      'Central AC compressor shut off completely and indoor temperature has reached 94F during Phoenix heat advisory; infant in home.'),
    STRUCT('REQ-5004', 'PROP-1005', DATE '2026-10-27', 'Inspector Mobile App', 'OPEN',
      'Front fascia gutter is detached and overflowing down exterior wall alongside a 3-foot diagonal crack in the stucco near foundation.'),
    STRUCT('REQ-5005', 'PROP-1001', DATE '2026-10-26', 'Resident Portal', 'OPEN',
      'Requesting routine autumn HVAC air filter replacement and seasonal exterior gutter leaf cleanout before winter.'),
    STRUCT('REQ-5006', 'PROP-1006', DATE '2026-10-26', 'Resident Portal', 'OPEN',
      'Kitchen dishwasher upper spray arm is clogged and leaves water spots on glasses; unit is not leaking onto floor.'),
    STRUCT('REQ-5007', 'PROP-1008', DATE '2026-10-29', 'After-Hours Hotline', 'OPEN',
      'Garage electrical subpanel is making a buzzing arcing sound and smells faintly like burnt plastic when dryer runs.'),
    STRUCT('REQ-5008', 'PROP-1011', DATE '2026-10-25', 'Resident Portal', 'OPEN',
      'Pantry bifold closet door slipped off its bottom plastic track guide; cosmetic convenience fix during regular hours.'),
    STRUCT('REQ-5009', 'PROP-1014', DATE '2026-10-29', 'After-Hours Hotline', 'OPEN',
      'Water heater tank in utility closet ruptured at bottom seam and is flooding hallway laminate flooring right now.'),
    STRUCT('REQ-5010', 'PROP-1017', DATE '2026-10-24', 'Resident Portal', 'OPEN',
      'Scheduled annual smoke and carbon monoxide detector battery inspection requested for all upstairs bedrooms.'),
    STRUCT('REQ-5011', 'PROP-1021', DATE '2026-10-28', 'Resident Portal', 'OPEN',
      'High winds peeled back metal ridge vent flashing on roof peak; loud flapping noise and daylight visible in attic.'),
    STRUCT('REQ-5012', 'PROP-1025', DATE '2026-10-23', 'Resident Portal', 'OPEN',
      'Master bathroom vanity drawer soft-close hinge is loose and guest bedroom baseboard needs minor touch-up paint.')
  ])
),
historical_resolved AS (
  SELECT
    FORMAT('REQ-%04d', 5100 + n) AS request_id,
    FORMAT('PROP-%04d', 1001 + MOD(n * 7, 180)) AS property_id,
    DATE_SUB(DATE '2026-10-20', INTERVAL MOD(n * 5, 320) DAY) AS request_date,
    ['Resident Portal', 'Inspector Mobile App'][OFFSET(MOD(n, 2))] AS reported_channel,
    'RESOLVED' AS status,
    ['Replaced 20x25 MERV-11 air filter during scheduled visit.',
     'Resealed kitchen sink silicone caulk bead and tested drain.',
     'Cleared pine needles from rear downspout elbow.',
     'Adjusted garage door photo-eye alignment sensor.'][OFFSET(MOD(n, 4))] AS tenant_description
  FROM UNNEST(GENERATE_ARRAY(1, 140)) AS n
)
SELECT * FROM curated_open_tickets
UNION ALL
SELECT * FROM historical_resolved;

-- 3. Regional Rent History (84 monthly points x 12 series = 1,008 rows for TimesFM AI.FORECAST)
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.regional_rent_history`
PARTITION BY month_date
CLUSTER BY metro_area, property_type
AS
WITH metros AS (
  SELECT * FROM UNNEST([
    STRUCT('Austin, TX' AS metro_area, 2150.0 AS base_rent, 0.0034 AS monthly_growth),
    STRUCT('Miami, FL', 2520.0, 0.0046),
    STRUCT('Denver, CO', 2280.0, 0.0031),
    STRUCT('Phoenix, AZ', 1980.0, 0.0037),
    STRUCT('Seattle, WA', 2610.0, 0.0039),
    STRUCT('Tampa, FL', 2060.0, 0.0043)
  ])
),
prop_types AS (
  SELECT * FROM UNNEST([
    STRUCT('Single-Family' AS property_type, 1.08 AS type_mult),
    STRUCT('Townhome' AS property_type, 0.91 AS type_mult)
  ])
),
months AS (
  SELECT
    m_idx,
    DATE_ADD(DATE '2019-01-01', INTERVAL m_idx MONTH) AS month_date
  FROM UNNEST(GENERATE_ARRAY(0, 83)) AS m_idx
)
SELECT
  mo.month_date,
  me.metro_area,
  pt.property_type,
  ROUND(
    (me.base_rent * pt.type_mult) * POW(1.0 + me.monthly_growth, mo.m_idx)
    + 58.0 * SIN(2.0 * ACOS(-1.0) * (EXTRACT(MONTH FROM mo.month_date) - 3) / 12.0)
    + (MOD(FARM_FINGERPRINT(CONCAT(me.metro_area, pt.property_type, CAST(mo.m_idx AS STRING))), 19) - 9) * 1.4,
    2
  ) AS avg_monthly_rent_usd,
  ROUND(
    0.942
    + 0.018 * SIN(2.0 * ACOS(-1.0) * (EXTRACT(MONTH FROM mo.month_date) - 4) / 12.0)
    + (MOD(ABS(FARM_FINGERPRINT(CONCAT('occ_', me.metro_area, CAST(mo.m_idx AS STRING)))), 9) - 4) * 0.001,
    4
  ) AS occupancy_rate
FROM months AS mo
CROSS JOIN metros AS me
CROSS JOIN prop_types AS pt;

-- 4. Property Valuation Comps (360 verified comparable leases for TabFM AI.PREDICT)
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.property_valuation_comps`
CLUSTER BY metro_area, property_type
AS
WITH base_comps AS (
  SELECT
    c_idx,
    FORMAT('COMP-%04d', 2000 + c_idx) AS comp_id,
    IF(c_idx <= 300, 'TRAIN', 'VALIDATION') AS split_set,
    ['Austin, TX', 'Miami, FL', 'Denver, CO', 'Phoenix, AZ', 'Seattle, WA', 'Tampa, FL'][OFFSET(MOD(c_idx, 6))] AS metro_area,
    ['Single-Family', 'Single-Family', 'Townhome', 'Small Multifamily'][OFFSET(MOD(c_idx, 4))] AS property_type,
    1200 + MOD(ABS(FARM_FINGERPRINT(CONCAT('csq_', CAST(c_idx AS STRING)))), 1950) AS sqft,
    2 + MOD(ABS(FARM_FINGERPRINT(CONCAT('cag_', CAST(c_idx AS STRING)))), 38) AS property_age_years,
    1 + MOD(ABS(FARM_FINGERPRINT(CONCAT('crf_', CAST(c_idx AS STRING)))), 20) AS roof_age_years,
    50 + MOD(ABS(FARM_FINGERPRINT(CONCAT('cwk_', CAST(c_idx AS STRING)))), 46) AS walkability_score,
    MOD(ABS(FARM_FINGERPRINT(CONCAT('ccm_', CAST(c_idx AS STRING)))), 6) AS prior_maintenance_claims_12m,
    (MOD(c_idx, 5) = 0) AS has_solar_upgrade
  FROM UNNEST(GENERATE_ARRAY(1, 360)) AS c_idx
),
enriched_comps AS (
  SELECT
    comp_id,
    split_set,
    metro_area,
    property_type,
    sqft,
    GREATEST(2, LEAST(5, CAST(ROUND(sqft / 650.0) AS INT64))) AS bedrooms,
    ROUND(1.5 + MOD(c_idx, 4) * 0.5, 1) AS bathrooms,
    property_age_years,
    roof_age_years,
    CASE
      WHEN metro_area = 'Denver, CO' THEN 'Moderate-Hail'
      WHEN metro_area IN ('Miami, FL', 'Tampa, FL') THEN 'High-Wind-Flood'
      WHEN metro_area = 'Phoenix, AZ' AND MOD(c_idx, 3) = 0 THEN 'Moderate-Wildfire'
      ELSE 'Low'
    END AS risk_zone,
    ROUND(0.15 + (roof_age_years / 32.0) + IF(metro_area IN ('Miami, FL', 'Tampa, FL', 'Denver, CO'), 0.16, 0.0), 2) AS hazard_risk_score,
    walkability_score,
    prior_maintenance_claims_12m,
    has_solar_upgrade,
    c_idx
  FROM base_comps
)
SELECT
  comp_id,
  split_set,
  metro_area,
  property_type,
  sqft,
  bedrooms,
  bathrooms,
  property_age_years,
  roof_age_years,
  risk_zone,
  hazard_risk_score,
  walkability_score,
  prior_maintenance_claims_12m,
  has_solar_upgrade,
  ROUND(
    (
      CASE metro_area
        WHEN 'Seattle, WA' THEN 2520
        WHEN 'Miami, FL' THEN 2630
        WHEN 'Denver, CO' THEN 2210
        WHEN 'Austin, TX' THEN 2120
        WHEN 'Tampa, FL' THEN 2040
        ELSE 1960
      END
      + CASE property_type
          WHEN 'Single-Family' THEN 180
          WHEN 'Small Multifamily' THEN 95
          ELSE -60
        END
      + sqft * 0.62
      + bedrooms * 65
      + bathrooms * 55
      + walkability_score * 3.2
      + IF(has_solar_upgrade, 95, 0)
      - roof_age_years * 9.5
      - hazard_risk_score * 160
      + (MOD(FARM_FINGERPRINT(comp_id), 41) - 20) * 2.5
    ),
    0
  ) AS market_monthly_rent_usd,
  IF(
    (prior_maintenance_claims_12m >= 4 AND roof_age_years >= 13)
    OR (hazard_risk_score >= 0.72 AND prior_maintenance_claims_12m >= 3),
    'CHURNED',
    'RENEWED'
  ) AS lease_renewal_outcome
FROM enriched_comps;
"""

bq_client.query_and_wait(SEED_SQL)
elapsed = time.perf_counter() - t0
print(f"✅ Created 4 Cotality tables in `{PROJECT_ID}.{DATASET_ID}` ({elapsed:.1f}s).\n")

# Verify row counts & storage metadata, plus run ML.DESCRIBE_DATA on properties
counts_sql = f"""
SELECT 'properties' AS table_name, COUNT(*) AS row_count FROM `{PROJECT_ID}.{DATASET_ID}.properties`
UNION ALL
SELECT 'maintenance_requests', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.maintenance_requests`
UNION ALL
SELECT 'regional_rent_history', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_history`
UNION ALL
SELECT 'property_valuation_comps', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.property_valuation_comps`
ORDER BY table_name;
"""
display(bq_client.query_and_wait(counts_sql).to_dataframe())

print("\n📋 Sample Featured Properties (Linked to Cloud Storage Inspection Images):")
sample_props_sql = f"""
SELECT
  property_id, metro_area, property_type, sqft, roof_material,
  roof_age_years, risk_zone, current_monthly_rent_usd, gcs_image_uri
FROM `{PROJECT_ID}.{DATASET_ID}.properties`
WHERE gcs_image_uri IS NOT NULL
ORDER BY property_id;
"""
display(bq_client.query_and_wait(sample_props_sql).to_dataframe())

print("\n🔎 Statistical Profile of `properties` via BigQuery `ML.DESCRIBE_DATA`:")
describe_sql = f"""
SELECT name, num_rows, num_nulls, min, max, mean, stddev
FROM ML.DESCRIBE_DATA(
  (SELECT sqft, roof_age_years, hazard_risk_score, current_monthly_rent_usd, estimated_asset_value_usd
   FROM `{PROJECT_ID}.{DATASET_ID}.properties`)
);
"""
display(bq_client.query_and_wait(describe_sql).to_dataframe())

---
## Step 3: Create Open Knowledge Format (OKF v0.2) Bundle for the Property Dataset

When AI agents or analysts query raw BigQuery schemas (`INFORMATION_SCHEMA.COLUMNS`) alone, they hit the **"Context Ceiling"**: column names like `current_monthly_rent_usd` and `estimated_asset_value_usd` look unambiguous, yet calculating Cotality's official **Net Effective Rent Yield** requires non-obvious institutional rules:
* Excluding `Vacant-Turn` parcels from stabilized yield rollups,
* Applying a `0.96` standard economic occupancy factor (`0.90` when `occupancy_status = 'Notice-Given'`), and
* Deducting `annual_operating_expense_usd` **plus** unmitigated roof/water hazard remediation reserves!

The **[Open Knowledge Format (OKF v0.2)](https://github.com/GoogleCloudPlatform/open-knowledge-format)** solves this by packaging human-and-agent-readable **Markdown + YAML frontmatter** files (`datasets/`, `tables/`, `metrics/`, `computations/`, `policies/`, `index.md`, `log.md`) that synchronize directly with **Google Cloud Knowledge Catalog**.

> **🎙️ Live Demo Talk Track (Step 3):**
> *"Before we unleash generative AI on our property tables, how do we make sure an AI agent doesn't invent its own financial math? Here in Step 3, we hydrate an Open Knowledge Format—OKF v0.2—bundle directly from BigQuery `INFORMATION_SCHEMA` combined with Cotality's underwriting policies and attested SQL computations. Watch the side-by-side comparison in Step 3.2: without OKF, an LLM computes a naive gross rent yield that overstates damaged properties; with OKF injected into `AI.GENERATE`, BigQuery applies the exact `Vacant-Turn` exclusion, occupancy allowance, and hazard capex reserve."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

import datetime
import re
import textwrap
import yaml

OKF_DIR = pathlib.Path("cotality_property_okf")
OKF_DIR.mkdir(parents=True, exist_ok=True)

# 1. Pull live column schemas from BigQuery INFORMATION_SCHEMA.COLUMNS
schema_df = bq_client.query_and_wait(f"""
SELECT table_name, column_name, data_type, ordinal_position
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('properties', 'maintenance_requests', 'regional_rent_history', 'property_valuation_comps')
ORDER BY table_name, ordinal_position
""").to_dataframe()


def format_schema_md_table(tbl_name: str) -> str:
    sub = schema_df[schema_df["table_name"] == tbl_name]
    lines = ["| Column | Data Type |", "| :--- | :--- |"]
    for _, r in sub.iterrows():
        lines.append(f"| `{r['column_name']}` | `{r['data_type']}` |")
    return "\n".join(lines)


now_iso = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
today_iso = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
stale_iso = "2027-12-31T00:00:00Z"
bq_ds_uri = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}"

okf_files = {
    "index.md": textwrap.dedent("""\
        ---
        okf_version: "0.2"
        title: "Cotality Property Intelligence Knowledge Bundle"
        description: "Open Knowledge Format (OKF v0.2) semantic layer governing Cotality property tables, metrics, attested SQL, and underwriting policies."
        ---

        # Cotality Property Intelligence — Open Knowledge Format (OKF v0.2)

        * [Datasets](datasets/) - Governed BigQuery dataset boundaries.
        * [Tables](tables/) - Partitioned and clustered property, maintenance, rent history, and valuation tables.
        * [Metrics](metrics/) - Canonical KPIs for Net Effective Rent Yield and Severe Hazard Exposure.
        * [Computations](computations/) - Attested BigQuery SQL computations.
        * [Policies](policies/) - Underwriting, 24-hour emergency dispatch, and lease renewal guardrails.
    """),
    "log.md": f"# Knowledge Bundle Change Log\n\n## {today_iso}\n\n* Generated and validated OKF v0.2 bundle from `{PROJECT_ID}.{DATASET_ID}`.\n",
    "datasets/index.md": "# Datasets\n\n* [Cotality Property AI Dataset](cotality_property_ai.md) - Unified BigQuery property dataset.\n",
    "tables/index.md": textwrap.dedent("""\
        # Tables

        * [Properties Master Table](properties.md) - Parcel attributes, hazard scores, rent, and Cloud Storage image URIs.
        * [Maintenance Requests](maintenance_requests.md) - Resident and inspector maintenance work orders.
        * [Regional Rent History](regional_rent_history.md) - 84-month metro rent time series for TimesFM forecasting.
        * [Property Valuation Comps](property_valuation_comps.md) - Comparable leases for TabFM zero-shot valuation.
    """),
    "metrics/index.md": "# Metrics\n\n* [Net Effective Rent Yield](net_effective_rent_yield.md) - Hazard-adjusted net property yield.\n",
    "computations/index.md": "# Attested Computations\n\n* [Net Effective Rent Yield SQL](net_effective_rent_yield_sql.md) - Verified BigQuery SQL implementation.\n",
    "policies/index.md": "# Policies\n\n* [Property Underwriting Policy](property_underwriting_policy.md) - Safety dispatch and renewal rent guardrails.\n",
    "datasets/cotality_property_ai.md": f"""---
type: "BigQuery Dataset"
title: "Cotality Property AI Dataset"
description: "Unified BigQuery dataset for structured property attributes, work orders, rent history, and multimodal inspection references."
resource: "{bq_ds_uri}"
status: "stable"
stale_after: "{stale_iso}"
generated:
  by: "cotality-okf-pipeline/0.2"
  at: "{now_iso}"
verified:
  - by: "human:cotality_data_steward"
    at: "{now_iso}"
sources:
  - id: "bq-schemata"
    resource: "{bq_ds_uri}/INFORMATION_SCHEMA.SCHEMATA"
    title: "BigQuery Dataset Catalog"
    author: "process:bigquery_information_schema"
    last_modified: "{now_iso}"
---

# Cotality Property AI Dataset

Contains [Properties Master Table](../tables/properties.md), [Maintenance Requests](../tables/maintenance_requests.md), [Regional Rent History](../tables/regional_rent_history.md), and [Property Valuation Comps](../tables/property_valuation_comps.md) [^bq-schemata].
""",
}

for tbl, title, desc in [
    ("properties", "Properties Master Table", "Parcel structural attributes, hazard scores, current lease rent, and Cloud Storage inspection URIs."),
    ("maintenance_requests", "Maintenance Requests", "Partitioned tenant and inspector natural-language maintenance tickets."),
    ("regional_rent_history", "Regional Rent History", "Monthly historical average rent across 6 metros and property types."),
    ("property_valuation_comps", "Property Valuation Comps", "Comparable property leases and renewal outcomes for TabFM AI.PREDICT."),
]:
    okf_files[f"tables/{tbl}.md"] = f"""---
type: "BigQuery Table"
title: "{title}"
description: "{desc}"
resource: "{bq_ds_uri}/tables/{tbl}"
status: "stable"
stale_after: "{stale_iso}"
generated:
  by: "cotality-okf-pipeline/0.2"
  at: "{now_iso}"
verified:
  - by: "human:cotality_data_steward"
    at: "{now_iso}"
sources:
  - id: "info-schema-{tbl}"
    resource: "{bq_ds_uri}/INFORMATION_SCHEMA.COLUMNS"
    title: "INFORMATION_SCHEMA.COLUMNS for {tbl}"
    author: "process:bigquery_information_schema"
    last_modified: "{now_iso}"
---

# {title} (`{tbl}`)

{desc} [^info-schema-{tbl}]

## Live BigQuery Schema

{format_schema_md_table(tbl)}
"""

okf_files["metrics/net_effective_rent_yield.md"] = f"""---
type: "Metric"
title: "Net Effective Rent Yield"
description: "Annualized net rental income after operating expenses and unmitigated hazard reserves divided by estimated asset value."
resource: "{bq_ds_uri}/metrics/net_effective_rent_yield"
status: "stable"
stale_after: "{stale_iso}"
generated:
  by: "cotality-okf-pipeline/0.2"
  at: "{now_iso}"
verified:
  - by: "human:cotality_data_steward"
    at: "{now_iso}"
sources:
  - id: "finance-charter"
    resource: "https://github.com/GoogleCloudPlatform/open-knowledge-format"
    title: "Cotality Portfolio Finance & Underwriting Charter"
    author: "human:cotality_risk_committee"
    last_modified: "{now_iso}"
---

# Net Effective Rent Yield

Measures stabilized hazard-adjusted return across [Properties Master Table](../tables/properties.md) [^finance-charter]:
* **Exclude** `occupancy_status = 'Vacant-Turn'` from stabilized rollups.
* **Economic Occupancy Factor**: `0.96` (`0.90` when `occupancy_status = 'Notice-Given'`).
* **Hazard Reserve Deduction**: Subtract `est_repair_cost_usd` when `urgent_remediation = TRUE`, else `1500` USD when `hazard_risk_score >= 0.70`.
* **Formula**: `100.0 * ((12 * current_monthly_rent_usd * occupancy_factor) - annual_operating_expense_usd - hazard_reserve_usd) / estimated_asset_value_usd`.
* **Attested SQL**: [Net Effective Rent Yield SQL](../computations/net_effective_rent_yield_sql.md).
"""

okf_files["computations/net_effective_rent_yield_sql.md"] = f"""---
type: "Attested Computation"
title: "Net Effective Rent Yield SQL"
description: "Verified BigQuery GoogleSQL computation for stabilized hazard-adjusted Net Effective Rent Yield."
resource: "{bq_ds_uri}/computations/net_effective_rent_yield_sql"
runtime: "bigquery"
status: "stable"
stale_after: "{stale_iso}"
generated:
  by: "cotality-okf-pipeline/0.2"
  at: "{now_iso}"
verified:
  - by: "human:cotality_data_steward"
    at: "{now_iso}"
sources:
  - id: "metric-ref"
    resource: "{bq_ds_uri}/metrics/net_effective_rent_yield"
    title: "Net Effective Rent Yield Specification"
    author: "human:cotality_risk_committee"
    last_modified: "{now_iso}"
---

# Net Effective Rent Yield SQL

Implements [Net Effective Rent Yield](../metrics/net_effective_rent_yield.md) [^metric-ref].

## Computation

```sql
SELECT
  metro_area,
  COUNT(*) AS stabilized_units,
  ROUND(
    100.0 * SAFE_DIVIDE(
      SUM(
        (12.0 * current_monthly_rent_usd * IF(occupancy_status = 'Notice-Given', 0.90, 0.96))
        - annual_operating_expense_usd
        - IF(hazard_risk_score >= 0.70, 1500.0, 0.0)
      ),
      SUM(estimated_asset_value_usd)
    ),
    2
  ) AS net_effective_rent_yield_pct
FROM `{PROJECT_ID}.{DATASET_ID}.properties`
WHERE occupancy_status != 'Vacant-Turn'
GROUP BY metro_area
ORDER BY net_effective_rent_yield_pct DESC;
```
"""

okf_files["policies/property_underwriting_policy.md"] = f"""---
type: "Policy"
title: "Property Underwriting Policy"
description: "Operational rules for 24-hour emergency maintenance dispatch, roof hazard capital reserves, and renewal rent increase caps."
resource: "{bq_ds_uri}/policies/property_underwriting_policy"
status: "stable"
stale_after: "{stale_iso}"
generated:
  by: "cotality-okf-pipeline/0.2"
  at: "{now_iso}"
verified:
  - by: "human:cotality_data_steward"
    at: "{now_iso}"
sources:
  - id: "policy-charter"
    resource: "https://github.com/GoogleCloudPlatform/open-knowledge-format"
    title: "Cotality Property Operations Governance Manual"
    author: "human:cotality_risk_committee"
    last_modified: "{now_iso}"
---

# Property Underwriting Policy

1. **Freeze Rent Increases on Unmitigated Hazards**: Any parcel with `urgent_remediation = TRUE` or an `OPEN` emergency ticket MUST cap renewal rent increases at `0%` until remediation is verified [^policy-charter].
2. **24-Hour Emergency Dispatch SLA**: Active roof/drywall water intrusion, electrical arcing, gas odor, or extreme-heat HVAC outages require `< 24h` vendor dispatch.
3. **Renewal Cap Guardrail**: Cap single-cycle mark-to-market lease renewal offers at `+8%` even when `AI.PREDICT` fair-market rent upside exceeds `8%`.
"""

# 2. Write locally, validate OKF v0.2 conformance rules, and sync to Cloud Storage
okf_errors = []
for rel_path, content in okf_files.items():
    local_p = OKF_DIR / rel_path
    local_p.parent.mkdir(parents=True, exist_ok=True)
    local_p.write_text(content, encoding="utf-8")

    # Structural OKF v0.2 checks
    if local_p.name == "log.md" and content.startswith("---"):
        okf_errors.append(f"{rel_path}: log.md must not have frontmatter")
    elif local_p.name == "index.md" and rel_path != "index.md" and content.startswith("---"):
        okf_errors.append(f"{rel_path}: subdirectory index.md must not have frontmatter")
    elif local_p.name not in ("index.md", "log.md"):
        m = re.fullmatch(r"---\s*\n(.*?)\n---\s*\n?(.*)", content, re.DOTALL)
        if not m:
            okf_errors.append(f"{rel_path}: missing valid YAML frontmatter")
        else:
            fm = yaml.safe_load(m.group(1))
            if not fm.get("type"):
                okf_errors.append(f"{rel_path}: missing required 'type'")
            if fm.get("type") == "Attested Computation" and not fm.get("runtime"):
                okf_errors.append(f"{rel_path}: Attested Computation missing 'runtime'")

    blob = bucket_obj.blob(f"{BUCKET_PREFIX}/okf/{rel_path}")
    blob.upload_from_filename(str(local_p), content_type="text/markdown")

assert not okf_errors, f"OKF validation errors: {okf_errors}"
print(f"✅ Generated, validated (0 errors), and uploaded {len(okf_files)} OKF v0.2 files to gs://{BUCKET_NAME}/{BUCKET_PREFIX}/okf/")
for rel_path in sorted(okf_files):
    print(f"   • {rel_path}")

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

okf_context_bundle = (
    (OKF_DIR / "metrics/net_effective_rent_yield.md").read_text(encoding="utf-8")
    + "\n---\n"
    + (OKF_DIR / "policies/property_underwriting_policy.md").read_text(encoding="utf-8")
)

# Compare an ungrounded schema-only prompt vs. an OKF-grounded prompt in BigQuery AI.GENERATE
okf_grounding_sql = f"""
WITH prompts AS (
  SELECT
    '1. Ungrounded (Raw Column Names Only)' AS mode,
    CONCAT(
      'Given table `{PROJECT_ID}.{DATASET_ID}.properties` with columns ',
      '(metro_area, occupancy_status, current_monthly_rent_usd, annual_operating_expense_usd, ',
      'hazard_risk_score, estimated_asset_value_usd), state the exact formula and filters you would ',
      'use to compute Net Effective Rent Yield (%) in 2 concise bullet points.'
    ) AS prompt_text
  UNION ALL
  SELECT
    '2. OKF v0.2 Grounded (Metric + Policy Spec)',
    CONCAT(
      'Using ONLY this governed Open Knowledge Format (OKF v0.2) specification, state the exact ',
      'formula, row exclusions, occupancy factors, and hazard reserve rules for Net Effective Rent Yield (%) ',
      'in 2 concise bullet points:\n\n',
      @okf_spec
    )
)
SELECT
  mode,
  AI.GENERATE(
    prompt_text{CONN_ARG_SQL}
  ).result AS generated_definition
FROM prompts
ORDER BY mode;
"""

job_cfg = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("okf_spec", "STRING", okf_context_bundle)]
)
df_okf_compare = bq_client.query_and_wait(okf_grounding_sql, job_config=job_cfg).to_dataframe()
for _, row in df_okf_compare.iterrows():
    print("=" * 80)
    print(f"📘 {row['mode']}")
    print("-" * 80)
    print(row["generated_definition"].strip())
print("\n📊 Attested OKF v0.2 Baseline Metric Result Across Metros:")
attested_yield_sql = f"""
SELECT
  metro_area,
  COUNT(*) AS stabilized_units,
  ROUND(AVG(current_monthly_rent_usd), 0) AS avg_monthly_rent_usd,
  ROUND(
    100.0 * SAFE_DIVIDE(
      SUM(
        (12.0 * current_monthly_rent_usd * IF(occupancy_status = 'Notice-Given', 0.90, 0.96))
        - annual_operating_expense_usd
        - IF(hazard_risk_score >= 0.70, 1500.0, 0.0)
      ),
      SUM(estimated_asset_value_usd)
    ),
    2
  ) AS okf_net_effective_yield_pct
FROM `{PROJECT_ID}.{DATASET_ID}.properties`
WHERE occupancy_status != 'Vacant-Turn'
GROUP BY metro_area
ORDER BY okf_net_effective_yield_pct DESC;
"""
display(bq_client.query_and_wait(attested_yield_sql).to_dataframe())

---
## Step 4: Multimodal Property & Roof Hazard Inspection (`OBJ.MAKE_REF` + `AI.GENERATE`)

Historically, extracting structured condition grades and repair reserve estimates from drone or field-inspector photos required exporting `gs://` paths out of the warehouse into external Python vision pipelines, then loading JSON results back into BigQuery.

With **`OBJ.MAKE_REF`** and **`AI.GENERATE`**, multimodal underwriting happens **directly inside a `SELECT` statement**:
1. **`OBJ.MAKE_REF(gcs_image_uri)`**: Constructs a typed `ObjectRef` `STRUCT<uri STRING, version STRING, authorizer STRING, details JSON>` pointing to the Cloud Storage object (and `OBJ.FETCH_METADATA` enriches `details.gcs_metadata` with MIME type, byte size, and MD5 hash). Using single-argument `OBJ.MAKE_REF(uri)` uses **Direct Access**, which works seamlessly inside **VPC Service Controls** perimeters without generating external HTTPS signed URLs.
2. **`AI.GENERATE(..., output_schema => ...)`**: Sends a multimodal `(text_instruction, ObjectRef)` tuple to **Gemini** and enforces a strongly typed BigQuery `STRUCT` schema (`condition_grade STRING`, `detected_hazards ARRAY<STRING>`, `urgent_remediation BOOL`, `est_repair_cost_usd INT64`, `underwriting_summary STRING`)—turning raw pixels into joinable SQL columns!

> **🎙️ Live Demo Talk Track (Step 4):**
> *"Now let's bring unstructured roof and interior inspection photos into the exact same SQL query as our parcel table. First, `OBJ.MAKE_REF` turns each `gs://` string into a governed BigQuery `ObjectRef` pointer and fetches live Cloud Storage metadata. Then, in a single `CREATE OR REPLACE TABLE` query, `AI.GENERATE` inspects all six property photos and returns strongly typed BigQuery columns—`BOOL`, `INT64`, `ARRAY<STRING>`—via `output_schema`. No brittle regexes, no JSON parsing scripts, and instant visibility into hail-damaged roofs and active ceiling leaks."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

import matplotlib.pyplot as plt
import seaborn as sns

print("🔗 Step 4.1: Inspecting Cloud Storage `ObjectRef` structs via `OBJ.MAKE_REF` + `OBJ.FETCH_METADATA`...")
obj_ref_sql = f"""
SELECT
  property_id,
  metro_area,
  roof_material,
  OBJ.FETCH_METADATA(OBJ.MAKE_REF(gcs_image_uri{MAKE_REF_CONN_SQL})) AS image_ref
FROM `{PROJECT_ID}.{DATASET_ID}.properties`
WHERE gcs_image_uri IS NOT NULL
ORDER BY property_id;
"""
df_obj_preview = bq_client.query_and_wait(f"""
WITH refs AS ({obj_ref_sql.rstrip(';')})
SELECT
  property_id,
  metro_area,
  image_ref.uri AS object_uri,
  JSON_VALUE(image_ref.details, '$.gcs_metadata.content_type') AS content_type,
  SAFE_CAST(JSON_VALUE(image_ref.details, '$.gcs_metadata.size') AS INT64) AS size_bytes
FROM refs
ORDER BY property_id;
""").to_dataframe()
display(df_obj_preview)

print("\n🔍 Step 4.2: Running Multimodal Property Underwriting via `OBJ.MAKE_REF` + `AI.GENERATE`...")
multimodal_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.property_image_assessments` AS
WITH inspected AS (
  SELECT
    p.property_id,
    p.metro_area,
    p.property_type,
    p.roof_material,
    p.roof_age_years,
    p.gcs_image_uri,
    AI.GENERATE(
      (
        CONCAT(
          'You are a senior Cotality property underwriting inspector. Analyze this field inspection image ',
          'and extract: (1) condition_grade as one of A, B, C, or D; (2) detected_hazards as concise visual ',
          'hazard tags (empty array if none); (3) urgent_remediation = TRUE if active water intrusion, missing ',
          'shingles, or exposed roof underlayment is present; (4) est_repair_cost_usd realistic USD remediation ',
          'cost (0 if intact/grade A); and (5) a 1-sentence underwriting_summary.'
        ),
        OBJ.MAKE_REF(p.gcs_image_uri{MAKE_REF_CONN_SQL})
      ),
      output_schema => 'condition_grade STRING, detected_hazards ARRAY<STRING>, urgent_remediation BOOL, est_repair_cost_usd INT64, underwriting_summary STRING'{CONN_ARG_SQL}
    ) AS ai_inspection
  FROM `{PROJECT_ID}.{DATASET_ID}.properties` AS p
  WHERE p.gcs_image_uri IS NOT NULL
)
SELECT
  property_id,
  metro_area,
  property_type,
  roof_material,
  roof_age_years,
  gcs_image_uri,
  ai_inspection.condition_grade,
  ai_inspection.detected_hazards,
  ai_inspection.urgent_remediation,
  ai_inspection.est_repair_cost_usd,
  ai_inspection.underwriting_summary,
  ai_inspection.status AS ai_status
FROM inspected;
"""
bq_client.query_and_wait(multimodal_sql)

df_inspections = bq_client.query_and_wait(f"""
SELECT
  property_id,
  metro_area,
  condition_grade,
  urgent_remediation,
  est_repair_cost_usd,
  ARRAY_TO_STRING(detected_hazards, ', ') AS detected_hazards_list,
  underwriting_summary
FROM `{PROJECT_ID}.{DATASET_ID}.property_image_assessments`
ORDER BY est_repair_cost_usd DESC, property_id;
""").to_dataframe()
display(df_inspections)

# Visualize Multimodal Estimated Repair Cost Exposure by Property & Urgency
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(11, 4.8))
bar_colors = ["#EA4335" if urg else "#34A853" for urg in df_inspections["urgent_remediation"]]
bars = ax.barh(
    df_inspections["property_id"] + " (" + df_inspections["metro_area"] + " | Grade " + df_inspections["condition_grade"].fillna("?") + ")",
    df_inspections["est_repair_cost_usd"].fillna(0),
    color=bar_colors,
)
ax.invert_yaxis()
ax.set_xlabel("AI-Estimated Remediation Reserve (USD)")
ax.set_title("Multimodal Inspection Output (`OBJ.MAKE_REF` + `AI.GENERATE`): Estimated Repair Cost by Parcel", fontweight="bold")
for bar in bars:
    val = bar.get_width()
    ax.text(val + 120, bar.get_y() + bar.get_height() / 2, f"${val:,.0f}", va="center", fontsize=9.5, fontweight="bold")
ax.set_xlim(0, max(df_inspections["est_repair_cost_usd"].max() * 1.22, 5000))
plt.tight_layout()
plt.show()

---
## Step 5: Intelligent Work-Order Triage & Vendor Dispatch (`AI.IF` + `AI.CLASSIFY`)

Property managers receive thousands of unstructured resident portal messages and inspector notes every week. Writing keyword `LIKE '%leak%'` rules breaks constantly (for example, *"dishwasher is **not** leaking"* triggers a false positive, while *"buzzing arcing sound and burnt plastic smell at breaker panel"* misses a `leak` filter completely!).

BigQuery provides **managed task-specific AI functions** that automatically optimize prompts, batching, and model selection under the hood:
* **`AI.IF(prompt, examples => ...)`**: Evaluates natural-language conditions and returns a native SQL `BOOL` (`TRUE` / `FALSE`)—usable directly in `SELECT`, `WHERE`, or `JOIN ON` predicates.
* **`AI.CLASSIFY(input, categories => ...)`**: Routes unstructured text or `ObjectRef` files into a controlled taxonomy of `(category, description)` pairs without prompt-engineering JSON schemas or cleaning up hallucinated label variants.

> **🎙️ Live Demo Talk Track (Step 5):**
> *"Look at ticket `REQ-5006`: the tenant explicitly says 'unit is not leaking onto floor.' A legacy regex for `%leak%` pages an emergency plumber at 2 AM by mistake, while missing `REQ-5007` where a garage electrical subpanel smells like burnt plastic! With `AI.IF` and `AI.CLASSIFY`, BigQuery understands semantic intent: `AI.IF` flags genuine 24-hour habitability emergencies as a native SQL `BOOL`, and `AI.CLASSIFY` routes every ticket into our five exact vendor trade queues."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

print("🚨 Triaging OPEN resident & inspector maintenance tickets with `AI.IF` + `AI.CLASSIFY`...")

triage_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.triaged_maintenance_requests` AS
SELECT
  m.request_id,
  m.property_id,
  p.metro_area,
  m.request_date,
  m.reported_channel,
  m.tenant_description,
  AI.IF(
    (
      CONCAT(
        'Does this property maintenance ticket describe an urgent 24-hour safety or habitability emergency ',
        '(such as active water intrusion/flooding, missing roof shingles/exposed underlayment, electrical arcing/burning odor, ',
        'or complete AC failure during extreme heat)? Ticket: '
      ),
      m.tenant_description
    ),
    examples => [
      ('Water is pouring from the ceiling drywall onto living room floor', TRUE),
      ('Dishwasher leaves spots on glasses; unit is not leaking onto floor', FALSE),
      ('Closet door came off bottom track; minor cosmetic repair', FALSE)
    ]{CONN_ARG_SQL}
  ) AS is_emergency_24h_dispatch,
  AI.CLASSIFY(
    m.tenant_description,
    categories => [
      ('Roof & Structural', 'Missing shingles, hail or wind roof damage, flashing, gutters, stucco or foundation cracks'),
      ('Plumbing & Water Intrusion', 'Active pipe leaks, ruptured water heaters, ceiling water stains, drains'),
      ('HVAC & Electrical', 'Air conditioning or heating outage, breaker panel buzzing/arcing, wiring hazards'),
      ('Appliance & Interior', 'Dishwasher, kitchen appliances, closet doors, cabinets, drywall touch-up paint'),
      ('Routine Preventive', 'Seasonal air filter replacement, gutter leaf cleanout, smoke detector battery check')
    ]{CONN_ARG_SQL}
  ) AS assigned_vendor_trade
FROM `{PROJECT_ID}.{DATASET_ID}.maintenance_requests` AS m
JOIN `{PROJECT_ID}.{DATASET_ID}.properties` AS p
  ON m.property_id = p.property_id
WHERE m.status = 'OPEN';
"""
bq_client.query_and_wait(triage_sql)

df_triage = bq_client.query_and_wait(f"""
SELECT
  request_id,
  property_id,
  metro_area,
  is_emergency_24h_dispatch,
  assigned_vendor_trade,
  tenant_description
FROM `{PROJECT_ID}.{DATASET_ID}.triaged_maintenance_requests`
ORDER BY is_emergency_24h_dispatch DESC, request_id;
""").to_dataframe()
display(df_triage)

# Plot Vendor Trade Routing Breakdown Stacked by 24h Emergency vs Standard SLA
trade_summary = (
    df_triage.assign(
        SLA_Tier=df_triage["is_emergency_24h_dispatch"].map(
            {True: "24-Hr Emergency Dispatch", False: "Standard Scheduled SLA"}
        )
    )
    .groupby(["assigned_vendor_trade", "SLA_Tier"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
cols_order = [c for c in ["24-Hr Emergency Dispatch", "Standard Scheduled SLA"] if c in trade_summary.columns]
palette_map = {"24-Hr Emergency Dispatch": "#EA4335", "Standard Scheduled SLA": "#4285F4"}
trade_summary[cols_order].plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color=[palette_map[c] for c in cols_order],
)
ax.set_xlabel("Open Work-Order Count")
ax.set_ylabel("Assigned Vendor Trade (`AI.CLASSIFY`)")
ax.set_title("Work-Order Triage (`AI.IF` Emergency Flag × `AI.CLASSIFY` Vendor Queue)", fontweight="bold")
ax.invert_yaxis()
ax.legend(title="Dispatch Priority (`AI.IF`)")
plt.tight_layout()
plt.show()

---
## Step 6: Zero-Shot Regional Rent Roll Forecasting (`AI.FORECAST`)

Portfolio asset managers need 12-month forward rent projections across every `(metro_area, property_type)` sub-market to guide acquisition underwriting and renewal budgets—without maintaining dozens of `ARIMA_PLUS` model objects.

BigQuery's **`AI.FORECAST`** table-valued function invokes Google's pre-trained **TimesFM** time-series foundation model **zero-shot**:
* Forecast **multiple series simultaneously** in one SQL query via `id_cols => ['metro_area', 'property_type']`.
* Validate out-of-time accuracy first on a chronological holdout (`2025` actuals vs. `2019–2024` history) using **`AI.EVALUATE`**.
* Return historical observations and 95% prediction intervals together via `output_historical_time_series => TRUE` and `confidence_level => 0.95`.

> **🎙️ Live Demo Talk Track (Step 6):**
> *"Notice there is no `CREATE MODEL` training step here! We first backtest TimesFM honestly on a 12-month chronological holdout using `AI.EVALUATE` to verify low error across all 12 metro and property-type series, and then run `AI.FORECAST` to project the next 12 months of Single-Family and Townhome rents—automatically capturing summer leasing seasonality and long-term metro growth."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

print("📏 Step 6.1: Chronological Backtest (`AI.EVALUATE` on 2019–2024 History vs. 2025 Actuals)...")
forecast_eval_sql = f"""
SELECT
  metro_area,
  property_type,
  ROUND(mean_absolute_error, 2) AS mae_usd,
  ROUND(mean_absolute_percentage_error, 2) AS mape_pct,
  ROUND(root_mean_squared_error, 2) AS rmse_usd
FROM AI.EVALUATE(
  (SELECT month_date, metro_area, property_type, avg_monthly_rent_usd
   FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_history`
   WHERE month_date < DATE '2025-01-01'),
  (SELECT month_date, metro_area, property_type, avg_monthly_rent_usd
   FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_history`
   WHERE month_date >= DATE '2025-01-01'),
  data_col => 'avg_monthly_rent_usd',
  timestamp_col => 'month_date',
  id_cols => ['metro_area', 'property_type'],
  horizon => 12
)
ORDER BY mape_pct ASC;
"""
df_fc_eval = bq_client.query_and_wait(forecast_eval_sql).to_dataframe()
display(df_fc_eval)

print("\n📈 Step 6.2: Generating 12-Month Multi-Metro Rent Forecast via `AI.FORECAST` (TimesFM)...")
forecast_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.regional_rent_forecast` AS
SELECT *
FROM AI.FORECAST(
  TABLE `{PROJECT_ID}.{DATASET_ID}.regional_rent_history`,
  data_col => 'avg_monthly_rent_usd',
  timestamp_col => 'month_date',
  id_cols => ['metro_area', 'property_type'],
  horizon => 12,
  confidence_level => 0.95,
  output_historical_time_series => TRUE
);
"""
bq_client.query_and_wait(forecast_sql)

df_forecast = bq_client.query_and_wait(f"""
SELECT
  metro_area,
  property_type,
  CAST(time_series_timestamp AS DATE) AS month_date,
  time_series_type,
  time_series_data,
  prediction_interval_lower_bound,
  prediction_interval_upper_bound
FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_forecast`
WHERE property_type = 'Single-Family'
  AND CAST(time_series_timestamp AS DATE) >= DATE '2023-01-01'
ORDER BY metro_area, month_date;
""").to_dataframe()

# Summarize projected 12-month YoY rent growth by metro for Single-Family
df_growth_summary = bq_client.query_and_wait(f"""
WITH hist_end AS (
  SELECT metro_area, property_type, time_series_data AS dec_2025_rent
  FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_forecast`
  WHERE time_series_type = 'history'
    AND CAST(time_series_timestamp AS DATE) = DATE '2025-12-01'
),
fc_end AS (
  SELECT metro_area, property_type, time_series_data AS dec_2026_forecast_rent
  FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_forecast`
  WHERE time_series_type = 'forecast'
    AND CAST(time_series_timestamp AS DATE) = DATE '2026-12-01'
)
SELECT
  h.metro_area,
  h.property_type,
  ROUND(h.dec_2025_rent, 0) AS dec_2025_actual_usd,
  ROUND(f.dec_2026_forecast_rent, 0) AS dec_2026_forecast_usd,
  ROUND(100.0 * (f.dec_2026_forecast_rent - h.dec_2025_rent) / h.dec_2025_rent, 2) AS projected_12m_growth_pct
FROM hist_end AS h
JOIN fc_end AS f USING (metro_area, property_type)
WHERE h.property_type = 'Single-Family'
ORDER BY projected_12m_growth_pct DESC;
""").to_dataframe()
display(df_growth_summary)

# Plot 2x3 grid of Single-Family Rent Forecasts with 95% Confidence Intervals
df_forecast["month_dt"] = pd.to_datetime(df_forecast["month_date"])
metros_sorted = sorted(df_forecast["metro_area"].unique())
fig, axes = plt.subplots(2, 3, figsize=(15.5, 8), sharey=False)
for ax, metro in zip(axes.flat, metros_sorted):
    sub = df_forecast[df_forecast["metro_area"] == metro]
    hist = sub[sub["time_series_type"] == "history"]
    fc = sub[sub["time_series_type"] == "forecast"]
    ax.plot(hist["month_dt"], hist["time_series_data"], color="#4285F4", linewidth=2, label="Historical Rent")
    ax.plot(fc["month_dt"], fc["time_series_data"], color="#34A853", linewidth=2.2, linestyle="--", label="TimesFM Forecast")
    ax.fill_between(
        fc["month_dt"],
        fc["prediction_interval_lower_bound"],
        fc["prediction_interval_upper_bound"],
        color="#34A853",
        alpha=0.20,
        label="95% Pred. Interval",
    )
    ax.set_title(f"{metro} (Single-Family)", fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("Avg Monthly Rent ($)")
axes[0, 0].legend(loc="upper left", fontsize=8.5)
plt.suptitle("Zero-Shot 12-Month Regional Rent Forecast (`AI.FORECAST` with TimesFM)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Step 7: Zero-Shot Automated Rent Valuation & Renewal Risk (`AI.PREDICT`)

> **ℹ️ Public Preview Note:** [`AI.PREDICT`](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-predict) is a **Public Preview** BigQuery table-valued function powered by **TabFM**, Google's pre-trained foundation model for structured tabular data (`<= 50` feature columns; `STRING`, `BOOL`, `INT64`, `FLOAT64`, `NUMERIC`, or `BIGNUMERIC`).

Traditionally, estimating parcel-level **fair-market monthly rent** (regression) and **tenant lease-renewal propensity** (classification) from comparable properties (`property_valuation_comps`) required feature encoding, hyperparameter tuning, `CREATE MODEL` training pipelines, and scheduled retraining jobs.

With **`AI.PREDICT`**, you pass your reference training query and target prediction query **at query time** via in-context tabular learning:
* **Numeric `label_col` (`market_monthly_rent_usd`)** $\rightarrow$ performs **zero-shot regression**, outputting `predicted_market_monthly_rent_usd` (evaluated via `AI.EVALUATE` / `ML.METRICS`).
* **Categorical `label_col` (`lease_renewal_outcome`)** $\rightarrow$ performs **zero-shot classification**, outputting `predicted_lease_renewal_outcome` and class probability array `predicted_lease_renewal_outcome_probs`!

> **🎙️ Live Demo Talk Track (Step 7):**
> *"This is usually the biggest 'aha' moment for data science and valuation teams. Look at the SQL in Step 7: there is zero `CREATE MODEL`, zero one-hot encoding, and zero hyperparameter tuning. We pass 300 comparable leases as context and ask `AI.PREDICT`—powered by Google's TabFM tabular foundation model—to price fair-market rent and predict lease-churn probability across our active portfolio in seconds. Immediately, we spot which legacy leases are \$250–\$350 below market!"*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

print("🎯 Step 7.1: Evaluating TabFM Zero-Shot Rent Valuation Accuracy on Validation Holdout (`AI.EVALUATE`)...")
tabfm_eval_sql = f"""
SELECT *
FROM AI.EVALUATE(
  (SELECT
     metro_area, property_type, sqft, bedrooms, bathrooms, property_age_years,
     roof_age_years, risk_zone, hazard_risk_score, walkability_score,
     prior_maintenance_claims_12m, has_solar_upgrade, market_monthly_rent_usd
   FROM `{PROJECT_ID}.{DATASET_ID}.property_valuation_comps`
   WHERE split_set = 'TRAIN'),
  (SELECT
     metro_area, property_type, sqft, bedrooms, bathrooms, property_age_years,
     roof_age_years, risk_zone, hazard_risk_score, walkability_score,
     prior_maintenance_claims_12m, has_solar_upgrade, market_monthly_rent_usd
   FROM `{PROJECT_ID}.{DATASET_ID}.property_valuation_comps`
   WHERE split_set = 'VALIDATION'),
  label_col => 'market_monthly_rent_usd'
);
"""
display(bq_client.query_and_wait(tabfm_eval_sql).to_dataframe().round(3))

print("\n💰 Step 7.2: Running Zero-Shot `AI.PREDICT` for Fair-Market Rent (Regression) & Renewal Risk (Classification)...")
predict_portfolio_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.property_ai_valuations` AS
WITH rent_predictions AS (
  SELECT
    property_id,
    address,
    metro_area,
    property_type,
    occupancy_status,
    sqft,
    current_monthly_rent_usd,
    ROUND(predicted_market_monthly_rent_usd, 0) AS predicted_market_rent_usd,
    ROUND(predicted_market_monthly_rent_usd - current_monthly_rent_usd, 0) AS monthly_rent_upside_usd,
    ROUND(100.0 * SAFE_DIVIDE(predicted_market_monthly_rent_usd - current_monthly_rent_usd, current_monthly_rent_usd), 1) AS rent_upside_pct
  FROM AI.PREDICT(
    -- Reference comparable leases (12 tabular features + numeric label)
    (SELECT
       metro_area, property_type, sqft, bedrooms, bathrooms, property_age_years,
       roof_age_years, risk_zone, hazard_risk_score, walkability_score,
       prior_maintenance_claims_12m, has_solar_upgrade, market_monthly_rent_usd
     FROM `{PROJECT_ID}.{DATASET_ID}.property_valuation_comps`
     WHERE split_set = 'TRAIN'),
    -- Active portfolio properties (all 12 features + passthrough identifier columns)
    (SELECT
       property_id, address, occupancy_status, current_monthly_rent_usd,
       metro_area, property_type, sqft, bedrooms, bathrooms, property_age_years,
       roof_age_years, risk_zone, hazard_risk_score, walkability_score,
       prior_maintenance_claims_12m, has_solar_upgrade
     FROM `{PROJECT_ID}.{DATASET_ID}.properties`),
    label_col => 'market_monthly_rent_usd'
  )
),
renewal_predictions AS (
  SELECT
    property_id,
    predicted_lease_renewal_outcome,
    ROUND(
      (SELECT prob FROM UNNEST(predicted_lease_renewal_outcome_probs) WHERE label = 'CHURNED'),
      3
    ) AS predicted_churn_probability
  FROM AI.PREDICT(
    (SELECT
       metro_area, property_type, sqft, bedrooms, bathrooms, property_age_years,
       roof_age_years, risk_zone, hazard_risk_score, walkability_score,
       prior_maintenance_claims_12m, has_solar_upgrade, lease_renewal_outcome
     FROM `{PROJECT_ID}.{DATASET_ID}.property_valuation_comps`
     WHERE split_set = 'TRAIN'),
    (SELECT
       property_id,
       metro_area, property_type, sqft, bedrooms, bathrooms, property_age_years,
       roof_age_years, risk_zone, hazard_risk_score, walkability_score,
       prior_maintenance_claims_12m, has_solar_upgrade
     FROM `{PROJECT_ID}.{DATASET_ID}.properties`),
    label_col => 'lease_renewal_outcome'
  )
)
SELECT
  r.*,
  c.predicted_lease_renewal_outcome,
  c.predicted_churn_probability
FROM rent_predictions AS r
JOIN renewal_predictions AS c USING (property_id);
"""
bq_client.query_and_wait(predict_portfolio_sql)

df_vals = bq_client.query_and_wait(f"""
SELECT
  property_id,
  metro_area,
  property_type,
  sqft,
  current_monthly_rent_usd,
  predicted_market_rent_usd,
  monthly_rent_upside_usd,
  rent_upside_pct,
  predicted_lease_renewal_outcome,
  predicted_churn_probability
FROM `{PROJECT_ID}.{DATASET_ID}.property_ai_valuations`
ORDER BY monthly_rent_upside_usd DESC;
""").to_dataframe()

print("🏆 Top 10 Under-Rented Value-Add Properties Surfaced by `AI.PREDICT`:")
display(df_vals.head(10))

# Plot Current In-Place Rent vs. TabFM Predicted Fair-Market Rent + Upside Distribution
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.5))

scatter_colors = df_vals["predicted_lease_renewal_outcome"].map({"RENEWED": "#4285F4", "CHURNED": "#EA4335"}).fillna("#5F6368")
axes[0].scatter(
    df_vals["current_monthly_rent_usd"],
    df_vals["predicted_market_rent_usd"],
    c=scatter_colors,
    alpha=0.75,
    edgecolors="white",
    s=55,
)
min_r = min(df_vals["current_monthly_rent_usd"].min(), df_vals["predicted_market_rent_usd"].min()) - 100
max_r = max(df_vals["current_monthly_rent_usd"].max(), df_vals["predicted_market_rent_usd"].max()) + 100
axes[0].plot([min_r, max_r], [min_r, max_r], linestyle="--", color="#202124", linewidth=1.4, label="Parity Line (In-Place = Market)")
axes[0].set_xlabel("Current Contracted Monthly Rent ($)")
axes[0].set_ylabel("TabFM Predicted Fair-Market Rent ($)")
axes[0].set_title("`AI.PREDICT` Valuation: In-Place vs. Fair-Market Rent (Red = Churn Risk)", fontweight="bold")
axes[0].legend(loc="upper left")

sns.boxplot(
    data=df_vals,
    y="metro_area",
    x="monthly_rent_upside_usd",
    hue="metro_area",
    legend=False,
    ax=axes[1],
    palette="Blues_r",
)
axes[1].axvline(0, color="#EA4335", linestyle="--", linewidth=1.3)
axes[1].set_xlabel("Monthly Mark-to-Market Rent Upside ($ / month)")
axes[1].set_ylabel("")
axes[1].set_title("Monthly Rent Upside Distribution by Metro (`AI.PREDICT`)", fontweight="bold")

plt.tight_layout()
plt.show()

---
## Step 8: Unified Portfolio Action Synthesis (`AI.GENERATE`) & Governance Guardrails

Finally, we bring **every signal together in a single BigQuery SQL query** over our 6 inspected flagship properties:
* **Multimodal Visual Condition** (`OBJ.MAKE_REF` + `AI.GENERATE` from Step 4),
* **Semantic Ticket Emergency & Vendor Trade** (`AI.IF` + `AI.CLASSIFY` from Step 5),
* **12-Month Regional Rent Forecast** (`AI.FORECAST` from Step 6),
* **Fair-Market Rent Upside & Churn Propensity** (`AI.PREDICT` from Step 7), and
* **Open Knowledge Format Policy Guardrails** (Step 3: *0% rent increase freeze on any parcel with urgent roof/water hazards, 24-hour vendor dispatch on emergencies, and a max `+8%` single-cycle renewal cap on clean properties*).

> **🎙️ Live Demo Talk Track (Step 8 — Closing Synthesis):**
> *"Here is where everything clicks for an executive audience. A raw valuation model alone might tell you to raise rent on `PROP-1002` in Denver because it's under market—right after a hailstorm tore 14 shingles off its roof! Because our multimodal inspection (`OBJ.MAKE_REF` + `AI.GENERATE`), resident ticket triage (`AI.IF` + `AI.CLASSIFY`), market outlook (`AI.FORECAST`), parcel valuation (`AI.PREDICT`), and OKF underwriting policy all live inside BigQuery, one SQL query automatically freezes rent increases on hazard-flagged parcels, dispatches 24-hour roofers and plumbers, and caps clean mark-to-market renewals at 8%."*

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

print("🧠 Synthesizing OKF-governed executive landlord action plans across all 6 multimodal-inspected parcels...")

portfolio_action_sql = f"""
WITH metro_growth AS (
  SELECT
    h.metro_area,
    ROUND(100.0 * (f.time_series_data - h.time_series_data) / h.time_series_data, 2) AS forecast_12m_rent_growth_pct
  FROM `{PROJECT_ID}.{DATASET_ID}.regional_rent_forecast` AS h
  JOIN `{PROJECT_ID}.{DATASET_ID}.regional_rent_forecast` AS f
    ON h.metro_area = f.metro_area
   AND h.property_type = f.property_type
  WHERE h.property_type = 'Single-Family'
    AND h.time_series_type = 'history' AND CAST(h.time_series_timestamp AS DATE) = DATE '2025-12-01'
    AND f.time_series_type = 'forecast' AND CAST(f.time_series_timestamp AS DATE) = DATE '2026-12-01'
),
open_ticket_rollup AS (
  SELECT
    property_id,
    LOGICAL_OR(is_emergency_24h_dispatch) AS has_emergency_ticket,
    STRING_AGG(CONCAT(assigned_vendor_trade, ': ', tenant_description), ' | ') AS open_tickets_summary
  FROM `{PROJECT_ID}.{DATASET_ID}.triaged_maintenance_requests`
  GROUP BY property_id
),
joined_signals AS (
  SELECT
    v.property_id,
    v.metro_area,
    v.current_monthly_rent_usd,
    v.predicted_market_rent_usd,
    v.rent_upside_pct,
    v.predicted_churn_probability,
    img.condition_grade,
    img.urgent_remediation,
    img.est_repair_cost_usd,
    img.underwriting_summary,
    COALESCE(t.has_emergency_ticket, FALSE) AS has_emergency_ticket,
    COALESCE(t.open_tickets_summary, 'No open tickets') AS open_tickets_summary,
    mg.forecast_12m_rent_growth_pct,
    -- Deterministic OKF Policy Guardrail calculation alongside AI narrative
    CASE
      WHEN img.urgent_remediation OR COALESCE(t.has_emergency_ticket, FALSE) THEN 0.0
      WHEN v.rent_upside_pct > 8.0 THEN 8.0
      WHEN v.rent_upside_pct > 0.0 THEN v.rent_upside_pct
      ELSE 0.0
    END AS okf_approved_renewal_increase_pct
  FROM `{PROJECT_ID}.{DATASET_ID}.property_ai_valuations` AS v
  JOIN `{PROJECT_ID}.{DATASET_ID}.property_image_assessments` AS img USING (property_id)
  LEFT JOIN open_ticket_rollup AS t USING (property_id)
  LEFT JOIN metro_growth AS mg USING (metro_area)
)
SELECT
  property_id,
  metro_area,
  condition_grade,
  urgent_remediation,
  has_emergency_ticket,
  est_repair_cost_usd,
  current_monthly_rent_usd,
  predicted_market_rent_usd,
  okf_approved_renewal_increase_pct,
  AI.GENERATE(
    CONCAT(
      'Write a crisp 2-sentence Cotality Property Manager action directive obeying OKF Policy ',
      '(0% rent cap if urgent_remediation or emergency ticket; <24h vendor dispatch on emergencies; ',
      'max +8% renewal cap on hazard-free under-rented homes). ',
      'Property ', property_id, ' in ', metro_area,
      ' | Visual Grade: ', IFNULL(condition_grade, 'N/A'),
      ' (', IFNULL(underwriting_summary, ''), ', Est Repair $', CAST(IFNULL(est_repair_cost_usd, 0) AS STRING), ')',
      ' | Open Tickets: ', open_tickets_summary,
      ' | Current Rent $', CAST(current_monthly_rent_usd AS STRING),
      ' vs TabFM Market Rent $', CAST(predicted_market_rent_usd AS STRING),
      ' | Metro 12m TimesFM Growth: ', CAST(IFNULL(forecast_12m_rent_growth_pct, 0.0) AS STRING), '%',
      ' | OKF Approved Renewal Adjustment: +', CAST(okf_approved_renewal_increase_pct AS STRING), '%.'
    ){CONN_ARG_SQL}
  ).result AS executive_action_plan
FROM joined_signals
ORDER BY property_id;
"""

df_actions = bq_client.query_and_wait(portfolio_action_sql).to_dataframe()
display(
    df_actions[
        [
            "property_id",
            "metro_area",
            "condition_grade",
            "urgent_remediation",
            "has_emergency_ticket",
            "est_repair_cost_usd",
            "current_monthly_rent_usd",
            "predicted_market_rent_usd",
            "okf_approved_renewal_increase_pct",
        ]
    ]
)

print("\n📋 Executive Landlord Directives (OKF Policy + Cross-Modal BigQuery AI):")
for _, r in df_actions.iterrows():
    badge = "🚨 HOLD RENT (0%) & DISPATCH" if (r["urgent_remediation"] or r["has_emergency_ticket"]) else f"✅ RENEWAL OFFER (+{r['okf_approved_renewal_increase_pct']:.1f}%)"
    print("=" * 88)
    print(f"{r['property_id']} ({r['metro_area']}) — {badge}")
    print("-" * 88)
    print((r["executive_action_plan"] or "").strip())

---
## Summary: End-to-End BigQuery AI/ML Architecture Takeaways

### Q&A
* **How do we analyze unstructured property inspection photos alongside structured parcel tables without external ETL pipelines?**
  By converting Cloud Storage URIs into governed `ObjectRef` values via **`OBJ.MAKE_REF`** (with metadata inspection via `OBJ.FETCH_METADATA`) and invoking **`AI.GENERATE`** with a strongly typed `output_schema`, BigQuery extracts structured condition grades, hazard arrays, and repair cost integers directly inside standard SQL queries—including inside VPC Service Controls perimeters via direct `ObjectRef` access.
* **How do we prevent generative AI queries from hallucinating financial KPIs or violating operational policies?**
  By authoring and validating an **Open Knowledge Format (OKF v0.2)** bundle (`datasets/`, `tables/`, `metrics/`, `computations/`, `policies/`) synchronized with **Google Cloud Knowledge Catalog**, ensuring every metric (`Net Effective Rent Yield`) and policy guardrail (`0%` rent freeze on hazard-flagged parcels, `+8%` single-cycle renewal cap) is deterministically grounded.

### Data Analysis Key Findings
* **Unified Multimodal & Semantic Triage (`OBJ.MAKE_REF`, `AI.GENERATE`, `AI.IF`, `AI.CLASSIFY`)**: Multimodal roof/interior inspection and natural-language ticket triage accurately isolate genuine 24-hour habitability emergencies (hail shingle loss, ceiling water intrusion, electrical arcing, extreme-heat AC outage) while avoiding false-positive emergency dispatches on negated keyword tickets (*"dishwasher is not leaking"*).
* **Zero-Shot Foundation Models (`AI.FORECAST` & `AI.PREDICT`)**: Built-in **TimesFM** (`AI.FORECAST`) projects seasonal 12-month rent trajectories across all 12 `(metro_area, property_type)` series in a single TVF call, while **TabFM** (`AI.PREDICT`, Public Preview) estimates parcel-level fair-market rent and tenant churn probability directly from comparable leases without `CREATE MODEL` training pipelines.

### Insights or Next Steps
* **Publish the OKF v0.2 Bundle to Knowledge Catalog**: Sync `cotality_property_okf/` to a version-controlled Git repository connected to Google Cloud Knowledge Catalog so Conversational Analytics agents and BI tools share the exact same attested SQL definitions.
* **Automate Continuous Event-Driven Triage**: Wrap the Step 4 (`OBJ.MAKE_REF` + `AI.GENERATE`) and Step 5 (`AI.IF` + `AI.CLASSIFY`) SQL statements into scheduled BigQuery queries or Dataform incremental tables to triage newly uploaded drone inspection photos and resident work orders automatically.

---
## Step 9: Optional Cleanup

If you wish to remove the lab BigQuery dataset (`cotality_property_ai`) and uploaded demo objects under `gs://{BUCKET_NAME}/cotality_property_demo/` after completing your workshop demonstration, set the cleanup flags to `True` and run the cell below.

In [ ]:
# Copyright 2026 Google LLC
# SPDX-License-Identifier: Apache-2.0

# @markdown ### 🧹 Clean Up Lab Resources
CONFIRM_DROP_DATASET = False  # @param {type:"boolean"}
CONFIRM_DELETE_GCS_DEMO_FILES = False  # @param {type:"boolean"}

if CONFIRM_DROP_DATASET:
    dataset_to_drop = f"{PROJECT_ID}.{DATASET_ID}"
    bq_client.delete_dataset(dataset_to_drop, delete_contents=True, not_found_ok=True)
    print(f"🗑️ Deleted BigQuery dataset `{dataset_to_drop}` and all its tables.")
else:
    print("ℹ️ Dataset cleanup skipped (set CONFIRM_DROP_DATASET = True to remove the lab dataset).")

if CONFIRM_DELETE_GCS_DEMO_FILES:
    blobs = list(bucket_obj.list_blobs(prefix=f"{BUCKET_PREFIX}/"))
    for b in blobs:
        b.delete()
    print(f"🗑️ Deleted {len(blobs)} demo object(s) under gs://{BUCKET_NAME}/{BUCKET_PREFIX}/.")
else:
    print("ℹ️ Cloud Storage cleanup skipped (set CONFIRM_DELETE_GCS_DEMO_FILES = True to delete demo objects).")